#House Price Prediction Model

##Importing Data

In [24]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

# Connect to database file and load the table
conn = sqlite3.connect(DATA_DIR / "HousePriceML.db")
df = pd.read_sql_query("SELECT * FROM Housing", conn)
conn.close()

In [25]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7229300521,20141013T000000,231300,2,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


##Data Clean-up

In [26]:
df = df.drop('id', axis=1)

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           21613 non-null  str    
 1   price          21613 non-null  int64  
 2   bedrooms       21613 non-null  int64  
 3   bathrooms      21613 non-null  float64
 4   sqft_living    21613 non-null  int64  
 5   sqft_lot       21613 non-null  int64  
 6   floors         21613 non-null  float64
 7   waterfront     21613 non-null  int64  
 8   view           21613 non-null  int64  
 9   condition      21613 non-null  int64  
 10  grade          21613 non-null  int64  
 11  sqft_above     21613 non-null  int64  
 12  sqft_basement  21613 non-null  int64  
 13  yr_built       21613 non-null  int64  
 14  yr_renovated   21613 non-null  int64  
 15  zipcode        21613 non-null  int64  
 16  lat            21613 non-null  float64
 17  long           21613 non-null  float64
 18  sqft_living15  21

In [28]:
df['date'] = df['date'].str.strip('"')
#df['floors'] = df['floors'].str.strip('"')
#df['zipcode'] = df['zipcode'].str.strip('"')

In [29]:
df['floors'] = df['floors'].astype(float)
df['zipcode'] = df['zipcode'].astype(int)

In [30]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

In [31]:
df['date'] = df['date'].dt.year

In [32]:
df[['bedrooms', 'bathrooms']] = round(df[['bedrooms', 'bathrooms']], 0)

##Feature Engineering

In [33]:
df['age'] = df['date'] - df['yr_built']
df['bed_per_bath'] = df['bedrooms'] / (df['bathrooms'] + 1)
df['time_since_renovated'] = df['date'] - df['yr_renovated']
df['rooms'] = df['bathrooms'] + df['bedrooms']
df['sqft_per_room'] = df['sqft_living'] / (df['rooms'] + 1)

In [34]:
df = df.iloc[:, [0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,1]]

In [35]:
df.head()

,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,lat,long,sqft_living15,sqft_lot15,age,bed_per_bath,time_since_renovated,rooms,sqft_per_room,price
0,2014,2,1.0,1180,5650,1.0,0,0,3,7,...,47.5112,-122.257,1340,5650,59,1.0,2014,3.0,295.000000,231300
1,2014,3,2.0,2570,7242,2.0,0,0,3,7,...,47.7210,-122.319,1690,7639,63,1.0,23,5.0,428.333333,538000
2,2015,2,1.0,770,10000,1.0,0,0,3,6,...,47.7379,-122.233,2720,8062,82,1.0,2015,3.0,192.500000,180000
3,2014,4,3.0,1960,5000,1.0,0,0,5,7,...,47.5208,-122.393,1360,5000,49,1.0,2014,7.0,245.000000,604000
4,2015,3,2.0,1680,8080,1.0,0,0,3,8,...,47.6168,-122.045,1800,7503,28,1.0,2015,5.0,280.000000,510000


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  21613 non-null  int32  
 1   bedrooms              21613 non-null  int64  
 2   bathrooms             21613 non-null  float64
 3   sqft_living           21613 non-null  int64  
 4   sqft_lot              21613 non-null  int64  
 5   floors                21613 non-null  float64
 6   waterfront            21613 non-null  int64  
 7   view                  21613 non-null  int64  
 8   condition             21613 non-null  int64  
 9   grade                 21613 non-null  int64  
 10  sqft_above            21613 non-null  int64  
 11  sqft_basement         21613 non-null  int64  
 12  yr_built              21613 non-null  int64  
 13  yr_renovated          21613 non-null  int64  
 14  zipcode               21613 non-null  int64  
 15  lat                   21613 no

##Dataset Split

In [37]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
Y = df['price']

In [38]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [39]:
x_train.info()

<class 'pandas.DataFrame'>
Index: 17290 entries, 17883 to 2347
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  17290 non-null  int32  
 1   bedrooms              17290 non-null  int64  
 2   bathrooms             17290 non-null  float64
 3   sqft_living           17290 non-null  int64  
 4   sqft_lot              17290 non-null  int64  
 5   floors                17290 non-null  float64
 6   waterfront            17290 non-null  int64  
 7   view                  17290 non-null  int64  
 8   condition             17290 non-null  int64  
 9   grade                 17290 non-null  int64  
 10  sqft_above            17290 non-null  int64  
 11  sqft_basement         17290 non-null  int64  
 12  yr_built              17290 non-null  int64  
 13  yr_renovated          17290 non-null  int64  
 14  zipcode               17290 non-null  int64  
 15  lat                   17290 non-

##XGBoost

In [40]:
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators = 1500, learning_rate = 0.06, max_depth = 5)

#Training
model.fit (x_train, y_train, eval_set=[(x_test, y_test)], verbose=100)

#Prediction
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {mse}")

[0]	validation_0-rmse:372956.57207
[100]	validation_0-rmse:137263.43009
[200]	validation_0-rmse:130281.09824
[300]	validation_0-rmse:128020.61269
[400]	validation_0-rmse:126653.58150
[500]	validation_0-rmse:125830.08170
[600]	validation_0-rmse:125432.28832
[700]	validation_0-rmse:124982.64069
[800]	validation_0-rmse:124693.53378
[900]	validation_0-rmse:124533.30382
[1000]	validation_0-rmse:124353.17799
[1100]	validation_0-rmse:124240.10243
[1200]	validation_0-rmse:124297.08355
[1300]	validation_0-rmse:124243.82046
[1400]	validation_0-rmse:124192.09479
[1499]	validation_0-rmse:124261.99457
Mean Squared Error: 15441043456.0


##Results


In [41]:
import numpy as np

results = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred,
    'Absolute Error': np.abs(y_test - y_pred),
    'Percentage Error': np.abs((y_test - y_pred) / y_test) * 100
})

formatted_results = results.copy()

formatted_results['Actual Price'] = formatted_results['Actual Price'].map('${:,.2f}'.format)
formatted_results['Predicted Price'] = formatted_results['Predicted Price'].map('${:,.2f}'.format)
formatted_results['Absolute Error'] = formatted_results['Absolute Error'].map('${:,.2f}'.format)
formatted_results['Percentage Error'] = formatted_results['Percentage Error'].map('{:,.2f}%'.format)

In [42]:
formatted_results.head(20)

,Actual Price,Predicted Price,Absolute Error,Percentage Error
4777,"$390,000.00","$396,070.84","$6,070.84",1.56%
12551,"$90,000.00","$244,421.20","$154,421.20",171.58%
7979,"$590,000.00","$651,056.06","$61,056.06",10.35%
17959,"$185,000.00","$194,326.20","$9,326.20",5.04%
7020,"$225,000.00","$224,192.39",$807.61,0.36%
17769,"$539,000.00","$558,894.12","$19,894.12",3.69%
18098,"$828,000.00","$799,900.75","$28,099.25",3.39%
15038,"$1,295,000.00","$1,435,901.88","$140,901.88",10.88%
13575,"$279,000.00","$253,183.31","$25,816.69",9.25%
10406,"$353,950.00","$387,940.75","$33,990.75",9.60%


In [43]:
total_percentage_error = np.mean(results['Percentage Error'])
median_ape = np.median(results['Percentage Error'])

print(f"Median Percentage Error: {median_ape:.2f}%")
print(f'Mean Percentage Error: {total_percentage_error:.2f}%')

Median Percentage Error: 7.63%
Mean Percentage Error: 12.12%


In [44]:
formatted_results['Actual Price'].str.strip("$")
formatted_results['Predicted Price'].str.strip("$")

4777       396,070.84
12551      244,421.20
7979       651,056.06
17959      194,326.20
7020       224,192.39
             ...     
10635      771,754.50
16253    1,285,231.50
8622       172,742.75
2509       535,767.25
20939    1,093,039.00
Name: Predicted Price, Length: 4323, dtype: str

In [45]:
formatted_results

,Actual Price,Predicted Price,Absolute Error,Percentage Error
4777,"$390,000.00","$396,070.84","$6,070.84",1.56%
12551,"$90,000.00","$244,421.20","$154,421.20",171.58%
7979,"$590,000.00","$651,056.06","$61,056.06",10.35%
17959,"$185,000.00","$194,326.20","$9,326.20",5.04%
7020,"$225,000.00","$224,192.39",$807.61,0.36%
...,...,...,...,...
10635,"$739,000.00","$771,754.50","$32,754.50",4.43%
16253,"$1,400,000.00","$1,285,231.50","$114,768.50",8.20%
8622,"$114,975.00","$172,742.75","$57,767.75",50.24%
2509,"$526,000.00","$535,767.25","$9,767.25",1.86%


In [46]:
formatted_results.to_csv("formatted_results.csv")